In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt


# Define how access classes in your data map to display labels
ACCESS_CONFIG = {
    "roaded": {
        "label": "Developed",
        "color": "#EE9A00",
        "plot_order": 1
    },
    "roadless": {
        "label": "IRA",
        "color": "#6E8B3D",
        "plot_order": 2
    },
    "wilderness": {
        "label": "Wilderness",
        "color": "#1874CD",
        "plot_order": 3
    }
}


In [ ]:
def load_extent_data(extent_csv_path: str) -> pd.DataFrame:
    """
    Load the extent data.

    Args:
        extent_csv_path (str): Path to the extent CSV. Must contain the columns
            state_name, access_class, and extent_masked_km2.

    Returns:
        pd.DataFrame: Columns state_name, access, and extent_masked_km2, with
            access lowercased and stripped.
    """
    df = pd.read_csv(extent_csv_path)
    df = df.rename(columns={"access_class": "access"})
    df["access"] = df["access"].str.lower().str.strip()
    return df[["state_name", "access", "extent_masked_km2"]]


In [ ]:
def _calculate_cumulative(df: pd.DataFrame, extent_dict: dict) -> pd.DataFrame:
    """
    Calculate cumulative burned percentage for each access class.

    Args:
        df (pd.DataFrame): Burned area records with columns year, access, and
            area_km2.
        extent_dict (dict): Access class mapped to its total extent in km2. Class
            keys come from ACCESS_CONFIG; missing keys fall back to an extent of
            1.

    Returns:
        pd.DataFrame: Columns year, access, and cumulative_pct, stacked across
            all access classes.
    """
    yearly = df.groupby(["year", "access"], as_index=False)["area_km2"].sum()
    
    cumulative = []
    for access_class in ACCESS_CONFIG.keys():
        data = yearly[yearly["access"] == access_class].sort_values("year")
        extent = extent_dict.get(access_class, 1)
        
        data["cumulative_pct"] = (data["area_km2"].cumsum() / extent) * 100
        cumulative.append(data[["year", "access", "cumulative_pct"]])
    
    return pd.concat(cumulative, ignore_index=True)


In [ ]:
def prepare_cumulative_data(
    df: pd.DataFrame,
    extent_df: pd.DataFrame,
    state_filter: str = "All"
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calculate cumulative burned proportion for total and high severity burns.

    Args:
        df (pd.DataFrame): Burned area records with columns year, mtbs_class,
            area_km2, access, and state_name. Rows outside MTBS classes 1-4, rows
            from 2024, and rows with missing values are dropped.
        extent_df (pd.DataFrame): Extent table with columns state_name, access,
            and extent_masked_km2.
        state_filter (str): State to restrict to, or "All" to sum extents across
            every state. Defaults to "All".

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Cumulative percentages for all
            severity classes and for high severity only (classes 3 and 4). Each
            has columns year, access, and cumulative_pct.
    """
    # Clean and filter data
    df = df.copy()
    df[["year", "mtbs_class", "area_km2"]] = df[["year", "mtbs_class", "area_km2"]].apply(pd.to_numeric, errors="coerce")
    df["access"] = df["access"].str.lower().str.strip()
    df = df.dropna(subset=["year", "mtbs_class", "area_km2", "access"])
    df = df[(df["mtbs_class"].isin([1, 2, 3, 4])) & (df["year"] != 2024)]
    
    # Apply state filter
    if state_filter != "All":
        df = df[df["state_name"] == state_filter]
        extent_df = extent_df[extent_df["state_name"] == state_filter]
    else:
        # Calculate "All" by summing across all states (matching notebook logic)
        extent_totals = extent_df.groupby("access", as_index=False)["extent_masked_km2"].sum()
        extent_totals["state_name"] = "All"
        extent_df = extent_totals
    
    extent_dict = extent_df.set_index("access")["extent_masked_km2"].to_dict()
    
    # Calculate cumulative for all severity and high severity
    cumulative_total = _calculate_cumulative(df, extent_dict)
    cumulative_high = _calculate_cumulative(df[df["mtbs_class"].isin([3, 4])], extent_dict)
    
    return cumulative_total, cumulative_high


In [ ]:
def _plot_panel(ax, data, title, show_ylabel=True):
    """
    Plot a single cumulative panel.

    Args:
        ax (matplotlib.axes.Axes): Axis to draw the panel on.
        data (pd.DataFrame): Cumulative data with columns year, access, and
            cumulative_pct. Access classes are drawn in ACCESS_CONFIG plot_order.
        title (str): Panel title.
        show_ylabel (bool): Whether to label the y-axis, used to suppress the
            label on interior panels. Defaults to True.

    Returns:
        None
    """
    # Sort access classes by plot_order for consistent legend ordering
    sorted_access = sorted(ACCESS_CONFIG.items(), key=lambda x: x[1]["plot_order"])
    
    for access_class, config in sorted_access:
        subset = data[data["access"] == access_class]
        if len(subset) > 0:
            ax.plot(
                subset["year"], 
                subset["cumulative_pct"],
                color=config["color"],
                linewidth=2.5,
                marker='o',
                markersize=4,
                label=config["label"]
            )
    
    ax.set_xlabel("Year", fontsize=16)
    if show_ylabel:
        ax.set_ylabel("Cumulative % of extent burned", fontsize=16)
    ax.set_title(title, fontsize=16)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.tick_params(axis='both', labelsize=14)
    ax.tick_params(axis='x', labelrotation=35)
    
    # Set x-ticks to years ending in 0 or 5, plus 2023
    if len(data) > 0:
        years = sorted(data["year"].unique())
        ticks = sorted(set([y for y in years if y % 5 == 0] + [2023]))
        ax.set_xticks(ticks)

    return None
    

In [ ]:
def plot_cumulative_comparison(
    cumulative_total: pd.DataFrame,
    cumulative_high_sev: pd.DataFrame,
    out_dir: str,
    state_label: str = "All"
    ) -> str:
    """
    Create side-by-side cumulative burn plots.

    Args:
        cumulative_total (pd.DataFrame): Cumulative data for all severity
            classes, with columns year, access, and cumulative_pct.
        cumulative_high_sev (pd.DataFrame): Cumulative data for moderate and high
            severity only, with the same columns.
        out_dir (str): Folder to save the figure to. Created if it does not
            exist.
        state_label (str): State name used in the output filename and to identify
            the subset plotted. Defaults to "All".

    Returns:
        str: Path to the saved PNG.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    
    # Set common y-axis limit
    y_max = max(
        cumulative_total["cumulative_pct"].max() if len(cumulative_total) > 0 else 0,
        cumulative_high_sev["cumulative_pct"].max() if len(cumulative_high_sev) > 0 else 0
    ) * 1.1
    
    _plot_panel(ax1, cumulative_total, "Total Area Burned")
    _plot_panel(ax2, cumulative_high_sev, "Moderate & High Severity Area Burned", show_ylabel=False)
    
    # Add panel labels
    ax1.text(0.02, 0.98, 'A', transform=ax1.transAxes, fontsize=48, 
             fontweight='bold', va='top', ha='left')
    ax2.text(0.02, 0.98, 'B', transform=ax2.transAxes, fontsize=48, 
             fontweight='bold', va='top', ha='left')
    
    ax1.set_ylim(0, y_max)
    ax2.set_ylim(0, y_max)
    ax2.legend(loc="upper right", frameon=False, fontsize=16)  # Changed from ax1 to ax2 and "upper left" to "upper right"
    
    # Save
    os.makedirs(out_dir, exist_ok=True)
    fname = f"mtbs_cumulative_comparison_{state_label.lower().replace(' ', '_')}.png"
    fpath = os.path.join(out_dir, fname)
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    
    return fpath


In [ ]:
def create_cumulative_plot(csv_path: str, extent_csv_path: str, out_dir: str) -> str:
    """
    Create comparison plot for all states combined.

    Args:
        csv_path (str): Path to the burned area CSV.
        extent_csv_path (str): Path to the extent CSV.
        out_dir (str): Folder to save the figure to.

    Returns:
        str: Path to the saved PNG.
    """
    df = pd.read_csv(csv_path)
    extent_df = load_extent_data(extent_csv_path)
    
    cumulative_total, cumulative_high_sev = prepare_cumulative_data(df, extent_df, "All")
    return plot_cumulative_comparison(cumulative_total, cumulative_high_sev, out_dir, "All")

## Run File I/O and logic

In [ ]:
# Define a path to the project folder
project_folder = "<PATH/TO/PROJECT/FOLDER>"

path = create_cumulative_plot(
    csv_path = os.path.join(project_folder, "data", "tables", "mtbs_nfs_access_class_summary.csv"),
    extent_csv_path = os.path.join(project_folder, "data", "tables", "nfs_access_extent_km2.csv"),
    out_dir = os.path.join(project_folder, "figures")
)

print(f"Created plot: {path}")